# Template Training Causal LM (Instruction Dataset)

Gunakan notebook ini sebagai alur latihan. Setiap bagian hanya berisi instruksi dan Anda mengisi kode di cell kosong di bawahnya.

Tujuan akhir: pipeline lengkap dari load data CSV, preprocessing, training, evaluasi (loss + perplexity), visualisasi, dan inference.

## 1. Import Library

Tulis semua import yang dibutuhkan. Minimal:
- `pandas`, `numpy`, `matplotlib`, `seaborn`
- `torch`, `torch.utils.data`
- `transformers` (misal `AutoTokenizer`, `AutoModelForCausalLM`, `DataCollatorForLanguageModeling`)
- `datasets` (misal `load_dataset`)

Tips: cek versi dengan `__version__` agar gampang debug.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils import data
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling
from datasets import load_dataset

print(pd.__version__)

2.3.3


## 2. Load Dataset (CSV) dan Inspect Kolom

Langkah yang perlu ditulis (gunakan pandas + datasets):
- Load CSV dengan `pandas.read_csv()` dan tampilkan `head()`.
- Tampilkan info kolom: `df.columns`, `df.info()`, `df.describe(include='all')`.
- Cek missing value: `df.isna().sum()`.
- Pastikan kolom `instruction`, `input`, `output` ada (atau kolom `text` jika sudah digabung).

Print yang disarankan:
- Jumlah baris dan kolom.
- Contoh 3 baris acak (`df.sample(3, random_state=42)`).

Jika ingin pakai `datasets`, gunakan `load_dataset('csv', data_files=...)` setelah CSV bersih.

In [2]:
df = pd.read_csv('./data/alpaca.csv')

display(df.head(3))

,instruction,input,output,text
0,Give three tips for staying healthy.,NaN,1.Eat a balanced diet and make sure to include...,Below is an instruction that describes a task....
1,What are the three primary colors?,NaN,"The three primary colors are red, blue, and ye...",Below is an instruction that describes a task....
2,Describe the structure of an atom.,NaN,"An atom is made up of a nucleus, which contain...",Below is an instruction that describes a task....


In [3]:
print(df.columns)

Index(['instruction', 'input', 'output', 'text'], dtype='object')


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52002 entries, 0 to 52001
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  52002 non-null  object
 1   input        20658 non-null  object
 2   output       51971 non-null  object
 3   text         52002 non-null  object
dtypes: object(4)
memory usage: 1.6+ MB


In [5]:
df.describe()

,instruction,input,output,text
count,52002,20658,51971,52002
unique,52002,19184,50840,52002
top,Analyze the given legal document and explain t...,No input,Negative,"Below is an instruction that describes a task,..."
freq,1,161,44,1


In [7]:
df.shape

(52002, 4)

In [6]:
df.isna().sum()

instruction        0
input          31344
output            31
text               0
dtype: int64

## 3. Preprocess: Gabungkan Instruction + Input + Output

Buat fungsi format teks agar konsisten. Contoh struktur:
```
Below is an instruction ...
### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
```

Instruksi detail:
- Buat fungsi `format_example(row)`.
- Jika `input` kosong/NaN, hilangkan blok Input.
- Simpan ke kolom baru `text`.
- Print contoh hasil format untuk 2-3 baris.

Cek distribusi panjang teks sebelum tokenisasi:
- Buat kolom `text_len` = `df['text'].str.len()`.
- Print statistik: `min`, `max`, `mean`, `median`.
- Visualisasi distribusi dengan `hist`/`seaborn.histplot`.

## 4. Tokenisasi dan Split Dataset

Langkah yang perlu ada:
- Load tokenizer: `AutoTokenizer.from_pretrained()`.
- Tentukan `max_length` (misal 128/256) berdasar distribusi panjang teks.
- Tokenisasi kolom `text` (gunakan `truncation=True`, `padding='max_length'`).
- Buat fungsi `tokenize_function(examples)` lalu `dataset.map(...)`.

Split dataset:
- Jika belum ada, buat split `train/valid/test` dengan `train_test_split()`.
- Print jumlah data per split.

Cek distribusi panjang token:
- Ambil `len(input_ids)` untuk beberapa contoh.
- Plot histogram panjang token.

## 5. Load Model & Tokenizer

Instruksi detail:
- Pilih model Causal LM kecil (misal `gpt2` atau `distilgpt2`).
- Load `AutoModelForCausalLM` dan `AutoTokenizer`.
- Set `tokenizer.pad_token = tokenizer.eos_token` jika perlu.
- Print konfigurasi singkat model: `model.config`.

Opsional: set `torch.device` dan pindahkan model ke device.

## 6. DataLoader & Collator

Instruksi detail:
- Gunakan `DataCollatorForLanguageModeling` (set `mlm=False`).
- Buat `DataLoader` untuk train dan valid.
- Pastikan batch berisi `input_ids`, `attention_mask`, dan `labels`.

Print debug:
- Ambil 1 batch dan print `batch.keys()` serta shape tensor.

## 7. Training Loop

Buat training loop manual:
- Inisialisasi optimizer (`AdamW`) dan learning rate.
- Loop per epoch dan per batch.
- `forward` -> `loss` -> `backward` -> `optimizer.step()`.
- Simpan `train_loss` per step/epoch ke list.

Print yang disarankan:
- Loss per beberapa step.
- Rata-rata loss per epoch.

## 8. Evaluasi (Loss & Perplexity)

Instruksi detail:
- Set `model.eval()`.
- Loop valid loader dengan `torch.no_grad()`.
- Hitung rata-rata `valid_loss`.
- Hitung perplexity: $\exp(\text{valid_loss})$.

Print:
- `valid_loss` dan `perplexity`.

## 9. Visualisasi Loss Curve

Instruksi detail:
- Plot `train_loss` dan `valid_loss` dengan matplotlib.
- Tambahkan label, judul, dan grid.
- Simpan plot jika perlu (`plt.savefig`).

## 10. Inference Setelah Training

Instruksi detail:
- Siapkan prompt singkat.
- Tokenisasi prompt, pindahkan ke device.
- Gunakan `model.generate()` dengan parameter dasar (`max_new_tokens`, `temperature`, `do_sample`).
- Decode hasil dan print.

Tips: bandingkan hasil sebelum vs sesudah training.